In [1]:
# Code for Table 4 and Appendix 3 in:
# Katajisto, K., & La Mela, M. (2026). Uncontested Transnational Identity: The Activation,
# Strengthening and Superiority of Nordic Identity in Finland's EU Debate of the 1990s.
# Scandinavian Journal of History.
#
# Data: ParliamentSampo speech dataset (CSV, 1980-2005), Linked Data Finland, https://www.ldf.fi, accessed in May 2026.
# Lemmatized speeches in the column "lemmas", see Appendix A
#
# Requirements: Python 3, pandas, scipy

In [2]:
import math
import urllib.request
from collections import Counter, defaultdict
import pandas as pd
from scipy.stats import chi2

In [4]:
# lemmatized speeches, source: www.ldf.fi
CORPUS_FILE = "./2026-speeches-csv-lemmas.csv"

# Finnish stopwords, source: https://github.com/stopwords-iso/stopwords-fi
STOPWORDS_URL = "https://raw.githubusercontent.com/stopwords-iso/stopwords-fi/master/stopwords-fi.txt"

# key word: lemmas beginning with "pohjoism" (Nordic*)
KEY_PREFIX = "pohjoism"

# window of +-5 lemmas within a speech, collocates with at least 20 co-occurrances
WINDOW = 5
MIN_COOCCURRENCE = 20

# periods (parliamentary years)
PERIODS = {
    "1980-1985": (1980, 1985),
    "1990-1995": (1990, 1995),
    "2000-2005": (2000, 2005),
}

# primary parties: at least 200 speeches, of which at least 20 mention nordic*
MIN_PARTY_SPEECHES = 200
MIN_PARTY_NORDIC_SPEECHES = 20

# context-specific terms removed in addition to the stopwords
FILTERED_TERMS = {
    "neuvosto",
    "yhteistyö",
    "pohjoismaa",
    "pohjoismainen",
    "valtuuskunta",
    "ministerineuvosto",
    "kokous",
    "istunto",
    "raportti",
    "kertomus",
    "suomi",
}

# party abbreviations harmonized for all periods (Centre Party with corrupted labels)
PARTY_NAMES = {
    "R": "RKP", "SD": "SDP",
    "SKL": "KD", "KD(VASTAUSPUHEENVUO)": "KD",
    "DEVA": "VAS", "SKDL": "VAS",
    "PS": "SMP",
    "KES": "KESK", "KESKI": "KESK", "KESKVASTAUSPUHEENVUORO)": "KESK",
}

In [7]:
# read lemmatized speeches
speeches = pd.read_csv(CORPUS_FILE, encoding="utf-8")

print(f"Speeches loaded: {len(speeches):,}")

# read stopwords, one word per line
with urllib.request.urlopen(STOPWORDS_URL) as response:
    stopword_text = response.read().decode("utf-8")

stopwords = set()
for line in stopword_text.splitlines():
    word = line.strip().lower()
    if word:
        stopwords.add(word)

# add the filtered terms
for term in FILTERED_TERMS:
    stopwords.add(term)

/var/folders/fl/300s02gj66j999hy0wwp4md40000gn/T/ipykernel_30873/2716354375.py:2: DtypeWarning: Columns (3,4,22,24) have mixed types. Specify dtype option on import or set low_memory=False.
  speeches = pd.read_csv(CORPUS_FILE, encoding="utf-8")


Speeches loaded: 168,206


In [8]:
# remove speeches without lemmas
speeches = speeches[speeches["lemmas"].notna() & (speeches["lemmas"].str.strip() != "")].copy()

# parliamentary year from the session id (part after "/")
speeches["year"] = speeches["session"].str.split("/").str[1].astype(int)

# period of each speech (speeches outside the periods are removed)
def get_period(year):
    for period, (start, end) in PERIODS.items():
        if start <= year <= end:
            return period
    return None

speeches["period"] = speeches["year"].apply(get_period)
speeches = speeches[speeches["period"].notna()].copy()

print(speeches.groupby("period").size())

# harmonize party abbreviations
speeches["party"] = speeches["party"].replace(PARTY_NAMES)


period
1980-1985    39280
1990-1995    62447
2000-2005    66478
dtype: int64


In [9]:
# tokens: split the lemma string, remove compound markers (#) and lowercase
def tokenize(lemmas):
    return [token.replace("#", "").lower() for token in lemmas.split()]

# true if the token is nordic*
def is_nordic(token):
    return token.startswith(KEY_PREFIX)

# tokens of each speech
speeches["tokens"] = speeches["lemmas"].apply(tokenize)

# true if the speech mentions nordic* at least once
speeches["mentions_nordic"] = speeches["tokens"].apply(lambda tokens: any(is_nordic(t) for t in tokens))


In [10]:
# primary parties: at least 200 speeches, of which at least 20 mention nordic*

primary_parties = {}
for period in PERIODS:
    period_speeches = speeches[speeches["period"] == period]
    speech_counts = period_speeches["party"].value_counts()
    nordic_counts = period_speeches[period_speeches["mentions_nordic"]]["party"].value_counts()
    primary_parties[period] = {
        party for party, n in speech_counts.items()
        if n >= MIN_PARTY_SPEECHES and nordic_counts.get(party, 0) >= MIN_PARTY_NORDIC_SPEECHES
    }
    print(f"{period}: {len(primary_parties[period])} primary parties: {sorted(primary_parties[period])}")

1980-1985: 7 primary parties: ['KD', 'KESK', 'KOK', 'RKP', 'SDP', 'SMP', 'VAS']
1990-1995: 9 primary parties: ['KD', 'KESK', 'KOK', 'LKP', 'RKP', 'SDP', 'SMP', 'VAS', 'VIHR']
2000-2005: 8 primary parties: ['KD', 'KESK', 'KOK', 'RKP', 'SDP', 'SMP', 'VAS', 'VIHR']


In [13]:
# co-occurrences: lemmas within +-5 lemmas of nordic* in the same speech
counts = {}
for period in PERIODS:
    period_speeches = speeches[speeches["period"] == period]

    # lemma frequencies in the period and in the nordic* windows
    corpus_freq = Counter()
    cooc_freq = Counter()

    # speeches and primary parties where the lemma co-occurs with nordic*
    cooc_speeches = defaultdict(set)
    cooc_parties = defaultdict(set)

    # occurrences of nordic* and number of tokens in all nordic* windows
    key_count = 0
    window_size = 0

    for speech_id, party, tokens in zip(period_speeches["speech_id"],
                                        period_speeches["party"],
                                        period_speeches["tokens"]):
        corpus_freq.update(tokens)
        for i, token in enumerate(tokens):
            if not is_nordic(token):
                continue
            key_count += 1
            start = max(0, i - WINDOW)
            end = min(len(tokens), i + WINDOW + 1)
            for j in range(start, end):
                if j == i:
                    continue
                collocate = tokens[j]
                window_size += 1
                cooc_freq[collocate] += 1
                cooc_speeches[collocate].add(speech_id)
                if party in primary_parties[period]:
                    cooc_parties[collocate].add(party)

    counts[period] = {
        "corpus_freq": corpus_freq,
        "cooc_freq": cooc_freq,
        "cooc_speeches": cooc_speeches,
        "cooc_parties": cooc_parties,
        "key_count": key_count,
        "window_size": window_size,
        "corpus_size": sum(corpus_freq.values()),
    }
    print(f"{period}: {counts[period]['corpus_size']:,} tokens, nordic* occurs {key_count:,} times")

1980-1985: 9,520,782 tokens, nordic* occurs 2,378 times
1990-1995: 15,488,817 tokens, nordic* occurs 4,771 times
2000-2005: 15,627,261 tokens, nordic* occurs 3,547 times


In [14]:
# log-likelihood (G2): collocate vs other tokens, inside vs outside the nordic* windows

def log_likelihood(o11, collocate_freq, window_size, corpus_size):
    o12 = window_size - o11
    o21 = collocate_freq - o11
    o22 = corpus_size - window_size - o21
    cells = [
        (o11, window_size, collocate_freq),
        (o12, window_size, corpus_size - collocate_freq),
        (o21, corpus_size - window_size, collocate_freq),
        (o22, corpus_size - window_size, corpus_size - collocate_freq),
    ]
    ll = 0.0
    for observed, row_total, col_total in cells:
        expected = row_total * col_total / corpus_size
        if observed > 0 and expected > 0:
            ll += observed * math.log(observed / expected)
    return 2 * ll


# collocates of nordic* in a period: MI, log-likelihood, significance
def collocation_table(period):
    c = counts[period]
    rows = []
    for collocate, o11 in c["cooc_freq"].items():
        if o11 < MIN_COOCCURRENCE or collocate in stopwords:
            continue
        collocate_freq = c["corpus_freq"][collocate]
        expected = c["window_size"] * collocate_freq / c["corpus_size"]
        mi = math.log2(o11 / expected)
        if mi <= 0:
            continue
        ll = log_likelihood(o11, collocate_freq, c["window_size"], c["corpus_size"])
        rows.append({
            "collocate": collocate,
            "MI": round(mi, 2),
            "LL": round(ll, 1),
            "p": chi2.sf(ll, df=1),
            "freq": o11,
            "n_speeches": len(c["cooc_speeches"][collocate]),
            "n_parties": len(c["cooc_parties"][collocate]),
        })
    table = pd.DataFrame(rows)

    # significant collocates: p < 0.05 with Bonferroni correction
    table["significant"] = table["p"] < (0.05 / len(table))
    return table.sort_values("MI", ascending=False).reset_index(drop=True)

In [15]:
# collocation tables per period
tables = {}
for period in PERIODS:
    tables[period] = collocation_table(period)

In [16]:
# table 4 (top 20) and appendix 3 (all significant) as tab-separated text
# max: highest value among the significant collocates in the period

def print_table(tables, top_n=None):
    for period in PERIODS:
        significant = tables[period][tables[period]["significant"]]
        subset = significant.head(top_n) if top_n else significant
        max_speeches = significant["n_speeches"].max()
        max_parties = significant["n_parties"].max()

        print(period.replace("-", "–") + "\t\t\t\t\t")
        print(f"collocate\tMI\tLL\tfreq\tspeeches (max {max_speeches})\tm_parties (max {max_parties})")
        for _, row in subset.iterrows():
            print(f"{row['collocate']}\t{row['MI']:.2f}\t{row['LL']:.1f}\t{row['freq']}\t"
                  f"{row['n_speeches']}\t{row['n_parties']}")

print("Table 4")
print_table(tables, top_n=20)

print("Appendix 3")
print_table(tables)

Table 4
1980–1985					
collocate	MI	LL	freq	speeches (max 86)	m_parties (max 7)
investointipankki	8.31	235.1	22	10	4
yhteiskuntajärjestelmä	6.21	266.3	39	38	1
suositus	4.78	180.8	38	21	5
tanska	4.56	133.8	30	27	7
vyöhyke	4.48	95.8	22	19	6
norja	4.38	139.0	33	33	7
ydinaseeton	4.37	88.2	21	16	5
pohjola	4.23	136.3	34	24	6
vapaa	4.07	258.7	68	61	4
demokratia	3.68	75.6	23	18	6
ruotsi	3.45	221.8	74	61	7
yk	3.14	72.7	28	20	6
yhteinen	3.13	149.7	58	41	7
etelä-afrikka	2.92	46.5	20	17	5
sopimus	2.73	126.2	60	43	7
välinen	2.52	68.7	37	25	6
taso	2.45	144.0	81	64	7
verrata	2.30	62.6	39	36	7
lainsäädäntö	2.27	84.7	54	45	7
kansainvälinen	2.26	84.2	54	42	7
1990–1995					
collocate	MI	LL	freq	speeches (max 302)	m_parties (max 9)
investointipankki	7.99	429.1	42	30	7
hyvinvointimalli	7.69	427.8	45	39	8
yhteiskuntamalli	7.29	191.1	22	19	8
hyvinvointivaltio	5.95	926.2	143	116	9
julkisuusperiaate	5.95	136.0	21	19	8
islanti	5.77	130.2	21	20	6
hyvinvointiyhteiskunta	5.41	568.0	100	81	9
baltia	4.35	234.4	56	47